# Imports and data prep

In [ ]:
from ddsp import DDSP, AudioDataset
from ddsp.synths import NoiseBandSynth, SubbandSineSynth

import torch
torch.set_float32_matmul_precision('medium')
torch.set_default_device('cuda')
torch.set_default_dtype(torch.float32)

from torch.utils.data import DataLoader, Subset

# Use preprocessed audio dataset (with the script from utils)
dataset_path = '/mnt/mariadata/datasets/ddsp-vae/birds/processed/'
sample_rate = 44100
duration = 2.0 # seconds
n_signal = sample_rate * duration
dataset = AudioDataset(dataset_path=dataset_path, n_signal=n_signal, sampling_rate=sample_rate)


# Split dataset into train and test sets
total_len = len(dataset)
val_len = int(0.2 * total_len)
indices = torch.randperm(total_len, generator=torch.Generator(device='cuda'))

val_indices = indices[:val_len]
train_indices = torch.arange(total_len, device='cuda')  # use full dataset for training

# Create subset datasets
train_set = Subset(dataset, train_indices)
val_set = Subset(dataset, val_indices)

# Create data loaders
batch_size = 16
lr = 1e-4
train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True, num_workers=0, generator=torch.Generator(device='cuda'))
val_loader = DataLoader(val_set, batch_size=batch_size, shuffle=False, num_workers=0, generator=torch.Generator(device='cuda'))


# Model initialization

In [ ]:
resampling_factor=32

nbs = NoiseBandSynth.to_config(n_filters=500, fs=sample_rate, resampling_factor=resampling_factor)
sss = SubbandSineSynth.to_config(n_sines=100, fs=sample_rate, resampling_factor=resampling_factor)


ddsp = DDSP(
  synth_configs=[nbs, sss],
  latent_size=4,
  fs=sample_rate,
  resampling_factor=resampling_factor,
  learning_rate=lr,

)

# Training

In [ ]:
import lightning as L

max_epochs = int(1e5)

trainer = L.Trainer(
  accelerator='cuda',
  max_epochs=max_epochs
)

trainer.fit(
  model=ddsp,
  train_dataloaders=train_loader,
  val_dataloaders=val_loader,
)


# Model examination

In [ ]:
from IPython.display import Audio, display

def examine_model(model, val_loader):
  """
  Examines the trained model by displaying original and autoencoded audio samples,
  as well as reporting on the loss.
  """
  model = model.cuda()
  model.eval()
  with torch.no_grad():
    x_audio = next(iter(val_loader))
    x_audio = x_audio.to('cuda')

    # Forward pass through the model
    y_audio = model(x_audio).squeeze(1)

    print(x_audio.shape, y_audio.shape)

    # Display original and reconstructed audio
    for j in range(x_audio.shape[0]):
      print(f"Sample {j + 1}:")
      x_audio_j = x_audio[j].cpu()
      y_audio_j = y_audio[j].cpu()

      display(Audio(x_audio_j.numpy(), rate=sample_rate))
      display(Audio(y_audio_j.numpy(), rate=sample_rate))

In [ ]:
examine_model(ddsp, val_loader)

# Export to nn~ compatible format

In [ ]:
# Export the model

training_path='/home/btadeusz/code/ddsp_vae/notebooks/lightning_logs/version_113/checkpoints'
synth_output_path='/home/btadeusz/code/ddsp_vae/notebooks/birds_test.ts'
!python -m cli.export --model_directory {training_path} --output_path {synth_output_path} --type last

print(f'Model saved at {synth_output_path}')